# DPO orca-math-korean

**Dataset:**
- https://huggingface.co/datasets/microsoft/orca-math-word-problems-200k
- https://huggingface.co/datasets/kuotient/orca-math-korean-dpo-pairs

**Model:**
- https://huggingface.co/soka0000/vclm-korean-7b

In [1]:
%pip install datasets transformers hf_transfer accelerate peft trl wandb scikit-learn -Uqq

Note: you may need to restart the kernel to use updated packages.


# 데이터 준비

In [2]:
from datasets import load_dataset # HuggingFace 데이터셋 로드 함수

# DPO용 학습 데이터 로드
dataset = load_dataset('kuotient/orca-math-korean-dpo-pairs', split='train')
SAMPLE_SIZE = 10000
dataset = dataset.select(range(SAMPLE_SIZE)) # 앞에서부터 10000개 데이터 선택
print(len(dataset))
print(dataset[100])



README.md:   0%|          | 0.00/2.08k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  162MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/192848 [00:00<?, ? examples/s]

10000
{'system': '당신은 인공지능 어시스턴트입니다.', 'question': '한 배럴에는 12리터(L)와 400밀리리터(ml)의 석유, B 배럴에는 7600밀리리터(ml)의 석유가 들어 있습니다. A 배럴과 B 배럴의 석유 양을 같게 하려면 A 배럴에서 B 배럴로 몇 리터(L)를 옮겨야 합니까?', 'chosen': '먼저 모든 측정값을 동일한 단위로 변환하여 계산을 쉽게 해봅시다. 모든 것을 밀리리터(ml)로 변환하겠습니다.\n\n배럴에는 12리터와 400밀리리터가 들어 있습니다. 1리터는 1000밀리리터와 같으므로 12리터를 밀리리터로 변환할 수 있습니다:\n12리터 = 12 * 1000밀리리터 = 12000밀리리터\n\n이제 이미 밀리리터 단위로 표시된 400밀리리터를 더합니다:\n12000밀리리터 + 400밀리리터 = 12400밀리리터\n\n따라서 배럴에는 총 12400밀리리터의 석유 가 들어 있습니다.\n\nB 배럴에는 7600밀리리터의 석유가 들어 있습니다.\n\n두 배럴의 석유 양을 동일하게 하려면 두 양의 평균을 구해야 합니다:\n두 배럴의 총량 = 12400밀리리터(A배럴) + 7600밀리리터(B배럴)\n두 배럴의 총량 = 20000밀리리터\n\n이제 이 총량을 2로 나누어 각 배럴에 해당하는 양을 구합니다:\n각 배럴의 동일한 양 = 20000밀리리터 / 2\n각 배럴의 동일한 양 = 10000밀리리터\n\n현재 A 배럴에는 12400밀리리터가 있으므로, 두 배럴의 양이 각각 10000밀리리터가 되도록 일부를 B 배럴로 이동해야 합니다.\n\nA에서 B로 이동할 양 = 12400밀리리터(A 배럴) - 10000밀리리터(동일한 양)\nA에서 B로 이동할 양 = 2400밀리리터\n\n따라서 두 배럴의 석유 양을 동일하게 하기 위해 A 배럴에서 B 배럴로 2400밀리리터(또는 2.4리터)를 이동해야 합니다.', 'rejected': ' 먼저 B 배럴의 석유 양을 리터로 변환하여 A 배럴의 석유 양과 비교해야 합니다.\n\nB 배럴의

## 모델 준비

https://huggingface.co/soka0000/vclm-korean-7b

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = 'soka0000/vclm-korean-7b'

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

The repository soka0000/vclm-korean-7b contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/soka0000/vclm-korean-7b .
 You can inspect the repository content at https://hf.co/soka0000/vclm-korean-7b.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


modeling_soka.py:   0%|          | 0.00/1.66k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/soka0000/vclm-korean-7b:
- modeling_soka.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/4.69k [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/2.52k [00:00<?, ?B/s]

In [4]:
import torch

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype = torch.bfloat16, # 메모리 절약 및 연산 효율성
    device_map = 'auto', # CPU / GPU 자동 배치
    trust_remote_code = True # 모델 저장소으 ㅣ커스텀 코드 허용
)

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

# 데이터 전처리

- Chat Template 적용
- prompt / chosen / rejected 형태로 변환

In [5]:
def preprocess_dpo_data(example):
    # Chat Template 적용 (system + user)
    messages = [
        {"role": 'system', 'content': example['system']},
        {"role": 'user', 'content': example['question']}
    ]
    
    # 모델이 기대하는 포맷으로 변환 (|im_start|system...|im_start|user...|)
    prompt_style = tokenizer.apply_chat_template(
        messages,
        tokenize = False,           # 토큰 ID 형태로 바꾸지 않고 문자열 형태로 변환
        add_generation_prompt=True  # 모델 답변 생성 프롬프트 형식 추가
    )
    
    # 템플릿 적용된 프롬프트 / 선호답변 / 비선호답변
    return {
        'prompt':prompt_style,
        'chosen':example['chosen'],
        'rejected': example['rejected']
    }
    
dataset_preprocessed = dataset.map(preprocess_dpo_data)        

print(f"변경전 컬럼: {dataset.column_names}")
print(f"변경 후 컬럼 : {dataset_preprocessed.column_names}")

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

변경전 컬럼: ['system', 'question', 'chosen', 'rejected']
변경 후 컬럼 : ['system', 'question', 'chosen', 'rejected', 'prompt']


In [6]:
dataset_preprocessed['prompt'][100]


'<|im_start|>system\n당신은 인공지능 어시스턴트입니다.<|im_end|>\n<|im_start|>user\n한 배럴에는 12리터(L)와 400밀리리터(ml)의 석유, B 배럴에는 7600밀리리터(ml)의 석유가 들어 있습니다. A 배럴과 B 배럴의 석유 양을 같게 하려면 A 배럴에서 B 배럴로 몇 리터(L)를 옮겨야 합니까?<|im_end|>\n<|im_start|>assistant\n'

In [7]:
# 전처리된 데이터셋 train / valid / test 데이터로 분할 (8 : 1 : 1) 
train_size = int(len(dataset_preprocessed)*0.8)
val_size = int(len(dataset_preprocessed)*0.1)
test_size = int(len(dataset_preprocessed)*0.1)

train_dataset = dataset_preprocessed.select(range(train_size)) # 앞 80%
val_dataset = dataset_preprocessed.select(range(train_size, train_size+val_size)) # 그 다음 10%
test_dataset = dataset_preprocessed.select(range(train_size + val_size, len(dataset_preprocessed))) # 나머지 10%

print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

8000
1000
1000


# BaseModel 학습 전 테스트

In [8]:
# 질문 (prompt) 를 넣고 모델이 답변을 생성하는 함수
def generate_response(model, tokenizer, question):
    prompt = question
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device) # prompt 토큰화
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,           # input_ids, attention_mask, ...
            max_length = 1024,  # 생성 토큰 포함 전체 토큰 길이
            do_sample = True,   # 창의적 생성
            top_k = 50,         # 상위 50개 토큰 중 선택
            top_p = 0.95,       # 누적확률 95% 이상인 토큰 중 선택
            temperature = 0.5,  # 약간 창의적
            num_return_sequences = 1, # 답변 1개
            eos_token_id  = tokenizer.eos_token_id, # 종료 토큰
            pad_token_id = tokenizer.pad_token_id   # 패딩 토큰
        )
        
        generated_text = tokenizer.decode(outputs[0]) # 생성된 토큰을 문자열로 디코딩
        return generated_text.replace(prompt, '').strip() # prompt 제거 후 답변만 반환
     

In [9]:
# 모델 학습 전 샘플 3개 테스트
test_subset = test_dataset.select(range(3))

for i, example in enumerate(test_subset):
    question = example['prompt']
    answer = example['chosen']
    
    print(f"질문 : {question} / 정답 : {answer}")
    generated_answer = generate_response(model, tokenizer, question)
    print(f"모델 생성 답변 : {generated_answer}")
    print("="*100)

질문 : <|im_start|>system
당신은 인공지능 어시스턴트입니다.<|im_end|>
<|im_start|>user
알리사와 아비게일은 과학 프로젝트를 위해 빈 캔 100개를 모아야 합니다. 오늘 현재 알리사는 빈 캔을 몇 개 모았고, 아비게일은 빈 캔 43개를 모았습니다. 그들은 빈 캔을 27개 더 모아야 합니다. 지금까지 알리사가 모은 빈 캔은 몇 개인가요?<|im_end|>
<|im_start|>assistant
 / 정답 : 알리사가 얼마나 많은 빈 캔을 모았는지 알아내려면, 아비게일이 모은 캔의 수와 프로젝트에 필요한 총 캔 수에서 아직 모아야 할 캔의 수를 빼야 합니다.

알리사의 캔 + 아비게일의 캔 + 아직 필요한 캔 = 필요한 총 캔 수입니다.
알리사의 캔 = 필요한 총 캔 수 - (아비게일의 캔 + 아직 필요한 캔)

우리는 이것을 알고 있습니다:
필요한 총 캔 수 = 100
아비게일의 캔 = 43
아직 필요한 캔 = 27

이제 숫자를 연결할 수 있습니다:

알리사의 캔 = 100 - (43 + 27)
알리사의 캔 = 100 - 70
알리사의 캔 = 30

알리사는 지금까지 빈 캔 30개를 모았습니다.
모델 생성 답변 : 주어진 정보를 분석해 보아요: - 알리사는 27개의 빈캔을 더 모아야 해요. - 아비게일은 43개의 빈캔을 모으셨어요. - 알리사가 지금까지 모은 빈캔의 수를 알아내야 해요. 알리사가 27개의 빈캔을 더 모아야 하므로, 그녀가 지금까지 모은 빈캔의 수는 총 필요한 빈캔 수에서 27을 뺀 값이에요. 이는 다음과 같이 나타낼 수 있어요: 알리사가 지금까지 모은 빈캔의 수 = 총 필요 빈캔 수 - 27 이제, 알리사가 지금까지 모은 빈캔의 수를 구하기 위해, 우리는 총 필요 빈캔 수를 알리사가 아비게일과 함께 모은 빈캔의 수에서 빼야 해요. 하지만, 총 필요 빈캔 수를 알리사가 아비게일과 함께 모은 빈캔의 수에서 빼야 한다는 것을 이해해야 해요. 하지만, 알리사가 아비게일과 함께 모은 빈캔의 수를 알리사가 지금까지 모은

# DOP 학습 준비

In [10]:
# LoRA : 전체 모델을 다 학습하지 않고, 일부 레이어만 저비용 학습
from peft import LoraConfig, get_peft_model # LoRa 설정 / 적용 함수

lora_config = LoraConfig(
    r = 8,              # 추가 학습할 저차원 행렬 크기
    lora_alpha= 32,     # LoRA 스케일 (업데이트 강도 조절)
    target_modules = ['q_proj', 'v_proj'], # LoRA 적용 모듈 (어텐션의 Q/V 프로젝션)
    lora_dropout=0.05, 
    bias = 'none',      # bias 학습하지 않음
    task_type='CAUSAL_LM' # 작업 유형 : 생성형 LM
) 

model = get_peft_model(model, lora_config) # 기존 모델에 LoRA 적용
model.print_trainable_parameters() # 학습 가능 파라미터 수 / 비율 확인

trainable params: 2,523,136 || all params: 7,618,139,648 || trainable%: 0.0331


In [11]:
# Reference (참조) 모델 : 정책모델이 얼마나 바뀌었는지 비교 기준이 되는 고정 모델
# chosen/rejected 답변에 대한 생성확률 비교값 제공

reference_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype = torch.bfloat16, # 메모리 절약 및 연산 효율성
    device_map = 'auto', # CPU / GPU 자동 배치
    trust_remote_code = True # 모델 저장소으 ㅣ커스텀 코드 허용
)

reference_model.eval() # 평가 모드로 전환 (dropout 등 비활성화)
for param in reference_model.parameters():
    param.requires_grad = False # 기울기 계산 / 업데이트 막음

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [12]:
import wandb

wandb.login() # 1 or 2 / API-Key / True

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice:  1


wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: After creating your account, create a new API key and store it securely.


wandb: Paste your API key and hit enter:  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: free-rjs103 (free-rjs103-palydata) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# DPO Model 테스트

In [13]:
# DPOConfig 설정 / DPOTrainer 학습 실행
from trl import DPOTrainer, DPOConfig

hub_model_id = 'kkkfu/vclm-korean-7b-orca-math-korean-dpo'

training_args = DPOConfig(  # DPO 학습 하이퍼파라미터/로깅/저장 설정
    output_dir='vclm-korean-7b-orca-math-korean-dpo',  # 체크포인트/로그 저장 폴더
    num_train_epochs=1,  # 전체 데이터를 1번 반복 학습
    per_device_train_batch_size=2,  # GPU 1개당 배치 크기
    gradient_accumulation_steps=4,  # 4번 누적 후 업데이트(실제 배치 효과: 2*4=8)
    learning_rate=5e-5,  # 학습률
    eval_strategy="steps",  # 일정 step마다 평가 수행
    save_strategy="steps",  # 일정 step마다 저장 수행
    logging_steps=50,  # 50 step마다 학습 로그 출력
    fp16=False,  # fp16 비활성화(여기서는 bf16 사용)
    bf16=True,  # bfloat16 사용(지원 GPU에서 안정적/빠름)
    tf32=True,  # Ampere 이상에서 matmul 가속 옵션(정밀도 약간 완화)
    beta=0.1,  # DPO의 beta(선호 강도 조절)
    max_length=512,  # prompt+답변을 포함한 최대 길이
    remove_unused_columns=False,  # DPO에 필요한 컬럼이 제거되지 않도록 유지
    push_to_hub=True,  # 학습 결과를 Hugging Face Hub로 업로드
    hub_model_id=hub_model_id,  # 업로드할 저장소 이름
    hub_strategy="end",  # 학습 끝난 뒤 한 번만 업로드
    report_to=['wandb']  # wandb로 학습 로그 전송
)

dpo_trainer = DPOTrainer(  # DPO Trainer 생성(정책모델 vs 참조모델 비교 학습)
    model=model,                 # 정책모델(LoRA 적용된 학습 대상)
    ref_model=reference_model,   # 참조모델(고정, 비교 기준)
    args=training_args,          # 위에서 만든 학습 설정
    train_dataset=train_dataset, # 학습 데이터(prompt/chosen/rejected)
    eval_dataset=val_dataset,    # 검증 데이터
    processing_class=tokenizer   # 토큰화 처리(TRL 버전에 따라 tokenizer 인자명 상이 가능)
)

dpo_trainer.train()  # DPO 학습 시작

Adding EOS to train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dropping fully truncated examples from eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
50,0.439941,0.147707,0.735886,287486.000000,1.216443,1.120583,0.836419,0.268769,-2.773075,0.976000,3.041844,-162.993608,-232.993889
100,0.060130,0.032691,0.856075,573651.000000,1.220769,1.113333,0.829359,-1.590764,-11.741764,0.989000,10.151000,-181.588943,-322.680779
150,0.041156,0.013288,0.874296,855914.000000,1.211278,1.098640,0.828334,-1.596653,-13.159162,0.994000,11.562509,-181.647826,-336.854758
200,0.023928,0.005861,0.869636,1143046.000000,1.188454,1.070605,0.828246,-1.410024,-14.243825,0.998000,12.833801,-179.781537,-347.701387
250,0.006023,0.005688,0.863555,1423484.000000,1.177715,1.061165,0.827088,-1.742973,-15.807025,0.999000,14.064052,-183.111032,-363.333388
300,0.019123,0.003228,0.848334,1716058.000000,1.183193,1.067836,0.827077,-1.703990,-16.268237,1.000000,14.564247,-182.721198,-367.945506
350,0.018127,0.003013,0.841025,2006203.000000,1.163740,1.044890,0.826387,-1.973730,-17.443537,1.000000,15.469807,-185.418602,-379.698512
400,0.016252,0.002548,0.853709,2290702.000000,1.148011,1.024169,0.826338,-1.917072,-17.896277,1.000000,15.979205,-184.852016,-384.225906
450,0.005635,0.002374,0.851466,2579757.000000,1.123588,0.996698,0.825793,-2.156254,-18.987731,1.000000,16.831477,-187.243838,-395.140448
500,0.033742,0.002703,0.836834,2865782.000000,1.135410,1.010021,0.827577,-1.801839,-17.842353,1.000000,16.040514,-183.699692,-383.686669


TrainOutput(global_step=1000, training_loss=0.03790994752570987, metrics={'train_runtime': 3961.1562, 'train_samples_per_second': 2.02, 'train_steps_per_second': 0.252, 'total_flos': 3.2174677370271744e+17, 'train_loss': 0.03790994752570987, 'epoch': 1.0})

In [14]:
# 학습을 중간에 끊을경우 수동 업로드

dpo_trainer.push_to_hub("Commit!")


CommitInfo(commit_url='https://huggingface.co/kkkfu/vclm-korean-7b-orca-math-korean-dpo/commit/9f748244da802af832e756a488b6d87edc8a6649', commit_message='Commit!', commit_description='', oid='9f748244da802af832e756a488b6d87edc8a6649', pr_url=None, repo_url=RepoUrl('https://huggingface.co/kkkfu/vclm-korean-7b-orca-math-korean-dpo', endpoint='https://huggingface.co', repo_type='model', repo_id='kkkfu/vclm-korean-7b-orca-math-korean-dpo'), pr_revision=None, pr_num=None)

# DPO 학습 후 Model 테스트

In [15]:
# 모델 학습 전 샘플 3개 테스트
test_subset = test_dataset.select(range(3))

for i, example in enumerate(test_subset):
    question = example['prompt']
    answer = example['chosen']

    print(f"질문 : {question} / 정답 : {answer}")

    generate_answer = generate_response(model, tokenizer, question)
    print(f"모델 생성 답변 : {generate_answer}")
    print("="*30)

질문 : <|im_start|>system
당신은 인공지능 어시스턴트입니다.<|im_end|>
<|im_start|>user
알리사와 아비게일은 과학 프로젝트를 위해 빈 캔 100개를 모아야 합니다. 오늘 현재 알리사는 빈 캔을 몇 개 모았고, 아비게일은 빈 캔 43개를 모았습니다. 그들은 빈 캔을 27개 더 모아야 합니다. 지금까지 알리사가 모은 빈 캔은 몇 개인가요?<|im_end|>
<|im_start|>assistant
 / 정답 : 알리사가 얼마나 많은 빈 캔을 모았는지 알아내려면, 아비게일이 모은 캔의 수와 프로젝트에 필요한 총 캔 수에서 아직 모아야 할 캔의 수를 빼야 합니다.

알리사의 캔 + 아비게일의 캔 + 아직 필요한 캔 = 필요한 총 캔 수입니다.
알리사의 캔 = 필요한 총 캔 수 - (아비게일의 캔 + 아직 필요한 캔)

우리는 이것을 알고 있습니다:
필요한 총 캔 수 = 100
아비게일의 캔 = 43
아직 필요한 캔 = 27

이제 숫자를 연결할 수 있습니다:

알리사의 캔 = 100 - (43 + 27)
알리사의 캔 = 100 - 70
알리사의 캔 = 30

알리사는 지금까지 빈 캔 30개를 모았습니다.
모델 생성 답변 : 먼저 주어진 정보를 분석해 보겠습니다.

1. 알리사와 아비게일이 총 모아야 할 빈캔 수: 100개
2. 아비게일이 이미 모은 빈캔 수: 43개
3. 두 사람이 더 모아야 할 빈캔 수: 27개

두 사람이 더 모아야 할 총 빈캔 수는 이미 모은 빈캔 수와 더하여 총 필요한 수보다 현재까지 모은 수를 구할 수 있습니다.

현재까지 모인 총 빈캔 수 = 이미 모은 빈캔 수 + 더 모아야 할 빈캔 수
= 43개 + 27개
= 70개

하지만 이것은 두 사람이 함께 모은 총량이며, 이 중 알리사의 기여분을 알아내려면 알리사가 이미 모은 빈캔 수를 알아야 합니다.

전체 빈캔 수(100개)에서 아비게일이 이미 모은 빈캔 수(43개)와 더해야 할 빈캔 수(27개)를 빼서 알리사가 더해야 할 빈캔 수를 구합니다:

알리사가 더

# DPO 성능 평가

In [16]:
# chosen vs rejected 선호 정확도 평가
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score

def calculate_log_prob(model, tokenizer, prompt, response):
    """  
    주어진 prompt 에 대한 responsse 의 log probability (평균) 값을 계산합니다.
    
    Args: 
        model: HuggingFace AutoModelForCausalLM (or similar)
        tokenizer : HuggingFace AutoTokenizer
        prompt (str) : 프롬프트 텍스트
        response (str) : 응답 텍스트
        
    Returns:
        float: response 토큰들의 평균 log probability    
    """
    
    full_text = prompt +" "+ response
    # 토큰화 후 장치 이동
    inputs = tokenizer(full_text, return_tensors='pt', truncation=True, max_length=512).to(model.device)
    input_ids = inputs['input_ids'] # 토큰 ID 들
    attention_mask = inputs['attention_mask'] # 패딩 무시용 마스크
    
    prompt_tokens = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(model.device) 
    prompt_len = prompt_tokens['input_ids'].shape[1] # 프롬프트 토큰 개수 확인 (response와의 경계)
    
    # 학습 안하고, 계산만 진행
    with torch.no_grad():
        outputs = model(input_ids = input_ids, attention_mask = attention_mask) # 순전파
        logits = outputs.logits   # 각 위치별 다음 토큰 후보들 점수
    
    # contiguous() : 텐서 데이터를 메모리상 연속된 형태로 재정리 함수    
    shift_logits = logits[..., :-1, :].contiguous() # 마지막 위치 제외(다음 토큰 예측용 길이 맞춰줌)
    shift_labels = input_ids[..., 1:].contiguous() # 정답 토큰을 한 칸 왼쪽으로 당김 (다음토큰)

    log_probs = F.log_softmax(shift_logits, dim=-1) # 점수표 ->log확률표(확률로 바꾼 뒤 log)

    # 각 위치별 정답 토큰 log 확률만 뽑아냄    
    true_log_probs = torch.gather(
        log_probs, # (batch, seq_len-1, vocab) -> (batch, seq_len -1, 1)
        2,           # vocab 차원에서 선택
        shift_labels.unsqueeze(-1)
    ).squeeze(-1) # (batch, seq_len -1, 1) -> (batch, seq_len -1)

    seq_len = shift_labels.shape[1] # 실제 점수 길이 (seq_len -1)
    # 프롬프트가 너무 길어서 response 구간이 없는 경우
    if prompt_len >= seq_len: 
        valid_log_probs = true_log_probs[:, -1:] # 마지막 값만 사용
    else:
        start_idx = max(0, prompt_len-1) # response 첫 토큰 점수 위치(프롬프트의 마지막 토큰 자리)
        valid_log_probs = true_log_probs[:, start_idx:] # prompt 점수 제외 response 점수만 사용
    
    avg_log_prob = valid_log_probs.mean().item() # response 점수 평균
    
    return avg_log_prob # 평균 log-prob

In [17]:
def calculate_preference_accuracy(model, tokenizer, dataset, num_samples = 100):
    """
    모델이 선호(chosen) 답변에 비선호(rejected) 답변보다 더 높은 확률을 부여하는지 평가 
    """
    
    correct = 0
    total = min(num_samples, len(dataset))
    
    print(f"전체 개수 : {total}")
    model.eval()
    
    for idx in range(total):
        example = dataset[idx]
        
        prompt = example['prompt']
        chosen = example['chosen']
        rejected = example['rejected']
        
        chosen_score = calculate_log_prob(model, tokenizer, prompt, chosen) # chosen 평균 log-prob
        rejected_score = calculate_log_prob(model, tokenizer, prompt, rejected) # rejected 평균 log-prob

        if chosen_score > rejected_score:
            correct +=1
            
        if (idx +1) % 10 == 0:
            print(f"{idx+1}/{total} 정확도 : {correct / (idx+1)*100:.2f}")
    
    accuracy = correct/total
    return accuracy

test_accuracy = calculate_preference_accuracy(model, tokenizer, test_dataset, num_samples=300)
print(test_accuracy) # chosen 점수가 더 높은 비율을 정확도로 출력 
        

전체 개수 : 300
10/300 정확도 : 60.00
20/300 정확도 : 55.00
30/300 정확도 : 70.00
40/300 정확도 : 67.50
50/300 정확도 : 66.00
60/300 정확도 : 61.67
70/300 정확도 : 61.43
80/300 정확도 : 65.00
90/300 정확도 : 67.78
100/300 정확도 : 68.00
110/300 정확도 : 67.27
120/300 정확도 : 68.33
130/300 정확도 : 68.46
140/300 정확도 : 68.57
150/300 정확도 : 68.00
160/300 정확도 : 68.75
170/300 정확도 : 69.41
180/300 정확도 : 68.89
190/300 정확도 : 70.53
200/300 정확도 : 72.00
210/300 정확도 : 72.38
220/300 정확도 : 72.73
230/300 정확도 : 73.04
240/300 정확도 : 74.17
250/300 정확도 : 74.00
260/300 정확도 : 74.62
270/300 정확도 : 74.81
280/300 정확도 : 74.29
290/300 정확도 : 74.83
300/300 정확도 : 74.33
0.7433333333333333
